# Week 20: Sensor-to-field learning with missing observations

Does set attention help when sensor cardinality changes, after accounting for representation and fitting budget?

**Prerequisites:** Week 19 sensing contract; Week 4/5 POD-DeepONet; Week 15 operator vocabulary.

Allow 120 minutes for the guided lab; estimate 3–5 hours total with reading and written analysis. This planning estimate has not been measured in a student pilot. Instructor solution edition.

In [ ]:
from pathlib import Path
import sys, tempfile, json
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'flowmllab/transformer_course.py').exists()),None)
if ROOT is None:
    raise RuntimeError('Extract the supplied course bundle and start Jupyter in its root; see docs/TRANSFORMER_COURSE.md for Colab upload setup.')
sys.path.insert(0,str(ROOT))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from flowmllab import transformer_course as lab
lab.seed_all(17)
cases,input_manifest=lab.load_data(ROOT)
EVIDENCE=ROOT/'results/transformer_course_v3'
# Any optional experiment must write into this disposable workspace.
workspace=tempfile.TemporaryDirectory(prefix='flowmllab-student-')
RUN_OUTPUT=Path(workspace.name)
task_status={}
print('Loaded checksummed simulated wake fields. No retained files will be rewritten.')

task_status={i:None for i in range(1,5)}


In [ ]:
RUN_EXERCISES=True

## Train a small set model during the laboratory

This 120-step classroom run is deliberately labeled a short optimization exercise. The release evidence uses the full protocol. Change the budget and examine the validation curve before comparing methods.

In [ ]:
from dataclasses import replace
p=replace(lab.Protocol(),max_steps=120,seeds=(17,))
training=np.concatenate([cases[r]['v'][::4] for r in p.train_re]);rep=lab.Representation.fit(training,8)
ids=lab.qr_sensors(rep,16);mean,scale=float(training.mean()),float(training.std())
x=lab.sensor_tokens(training,ids,cases[90],mean,scale);y=rep.encode(training)
vx=lab.sensor_tokens(cases[100]['v'][::4],ids,cases[90],mean,scale);vy=rep.encode(cases[100]['v'][::4])
model=lab.SensorSet();record=lab.fit(model,x,y,lambda m:np.mean((lab.infer(m,vx)-vy)**2),p)
print({k:v for k,v in record.items() if k!='history'})
plt.plot([r['step'] for r in record['history']],[r['validation_mse'] for r in record['history']]);plt.xlabel('optimizer steps');plt.ylabel('validation coefficient MSE');plt.show()

## Audit the retained all-sensor and dropout comparisons

The set model was trained with cardinality augmentation. Fixed-input networks and ridge use declared training-mean imputation under dropout. Gappy POD solves directly on the available coordinates. This is a system comparison, not a perfectly capacity-matched architecture ablation. A separate three-seed paired ablation keeps SensorSet, initialization, observations and selection protocol fixed while changing cardinality augmentation. It shows a tradeoff: better fixed half-sensor error but worse all-sensor error in all three pairs. This is a training-recipe effect, not an architecture-superiority claim.

In [ ]:
record=json.loads((EVIDENCE/'sensors.json').read_text())
table=[{'method':r['method'],'seed':r['seed'],'condition':r['condition'],'field':r['metrics']['field_relative_l2'],'floor':r['metrics']['representation_floor'],'subspace':r['metrics']['in_subspace_relative_l2'],'boundary':r.get('training',{}).get('at_budget_boundary')} for r in record['rows']]
display(pd.DataFrame(table))
retained=np.load(EVIDENCE/'predictions.npz',allow_pickle=False)
row=record['rows'][0];pred=lab.prediction(retained,'sensors__'+row['key'])
truth64=np.asarray(cases[105]['v'],dtype=np.float64).reshape(281,-1)
recomputed=np.linalg.norm(np.asarray(pred,dtype=np.float64)-truth64)/np.linalg.norm(truth64)
assert np.isclose(recomputed,row['metrics']['field_relative_l2'],rtol=1e-6,atol=1e-9)
print('Rescored retained prediction:',recomputed)
ablation=json.loads((ROOT/"results/transformer_sensor_ablation/metrics.json").read_text())
display(pd.DataFrame(ablation["rows"]))
print("Three paired seeds are descriptive; selection uses all-sensor validation and executed budgets can differ.")

## Task 1: Consistent sensor-token dropout

Remove every other sensor from values and coordinates together; never reinterpret a missing value as an observed zero.

Interface contract: Input tokens:[batch,sensors,features] is a NumPy array. Return tokens[:,::2,:], preserving all features for every retained sensor.

In [ ]:
def drop_half(tokens):
    return tokens[:,::2,:]

In [ ]:
if RUN_EXERCISES:
    try:
        for n in (5,16):
            tokens=np.arange(2*n*3).reshape(2,n,3)
            np.testing.assert_array_equal(drop_half(tokens),tokens[:,::2,:])
        task_status[1]=True
    except NotImplementedError:
        task_status[1]=None
    except Exception as exc:
        task_status[1]=False
        print("Task 1 check failed:",type(exc).__name__,str(exc))
else:
    task_status[1]=None
print("Task 1:","PASS" if task_status[1] is True else "NOT SUBMITTED" if task_status[1] is None else "FAILED")

## Task 2: Capacity accounting

Count trainable parameters; compare the attention model with the retained 16-208-8 branch network (Protocol.branch_width=208).

Interface contract: Input is a torch.nn.Module. Return a Python integer counting only parameters with requires_grad=True, including biases.

In [ ]:
def parameter_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
if RUN_EXERCISES:
    try:
        for a,b in ((3,2),(7,4)):
            net=torch.nn.Linear(a,b)
            assert parameter_count(net)==a*b+b
            net.bias.requires_grad_(False)
            assert parameter_count(net)==a*b
        task_status[2]=True
    except NotImplementedError:
        task_status[2]=None
    except Exception as exc:
        task_status[2]=False
        print("Task 2 check failed:",type(exc).__name__,str(exc))
else:
    task_status[2]=None
print("Task 2:","PASS" if task_status[2] is True else "NOT SUBMITTED" if task_status[2] is None else "FAILED")

## Task 3: Training budget diagnosis

Flag a record whose selected checkpoint reaches its ceiling. This diagnoses an unresolved budget limit, not proof of convergence or its absence.

Interface contract: record is a dictionary with integer selected_step and ceiling. Return True exactly when selected_step==ceiling; a near-ceiling checkpoint is False.

In [ ]:
def boundary_flag(record):
    return record["selected_step"]==record["ceiling"]

In [ ]:
if RUN_EXERCISES:
    try:
        assert boundary_flag({"selected_step":120,"ceiling":120})
        assert not boundary_flag({"selected_step":90,"ceiling":120})
        for step in (0,99,100,101,119):
            assert not boundary_flag({'selected_step':step,'ceiling':120})
        assert boundary_flag({'selected_step':0,'ceiling':0})
        task_status[3]=True
    except NotImplementedError:
        task_status[3]=None
    except Exception as exc:
        task_status[3]=False
        print("Task 3 check failed:",type(exc).__name__,str(exc))
else:
    task_status[3]=None
print("Task 3:","PASS" if task_status[3] is True else "NOT SUBMITTED" if task_status[3] is None else "FAILED")

## Task 4: Implement validation-only checkpoint selection

Train a supplied small model using full-batch SGD and mean squared error. Evaluate validation at step zero and after each update; retain and restore the best state. Return selected step, executed steps and validation trace. Neither evaluation data nor evaluation loss is an input. Compare two learning rates and explain any step-zero selection.

Interface contract: steps is a nonnegative integer, lr a positive scalar. Inputs are dimension-compatible Torch tensors. Use full-batch SGD with mean MSE. Evaluate step zero and every update; select the earliest minimum validation loss; restore it and leave model in evaluation mode. Return dictionary selected_step (int), executed_steps (steps), validation_mse (float minimum), trace (list of dictionaries with step and validation_mse for 0..steps). Negative steps or nonpositive lr must raise ValueError.

In [ ]:
def fit_with_validation(model,train_x,train_y,val_x,val_y,steps,lr):
    import copy
    if steps<0 or lr<=0:raise ValueError('invalid budget or learning rate')
    optimizer=torch.optim.SGD(model.parameters(),lr=lr)
    best_state=None
    best_loss=float('inf')
    selected=0
    trace=[]
    for step in range(steps+1):
        model.eval()
        with torch.no_grad():
            loss=float(((model(val_x)-val_y)**2).mean())
        trace.append({'step':step,'validation_mse':loss})
        if loss<best_loss:
            best_loss=loss
            selected=step
            best_state=copy.deepcopy(model.state_dict())
        if step==steps:break
        model.train()
        optimizer.zero_grad()
        train_loss=((model(train_x)-train_y)**2).mean()
        train_loss.backward()
        optimizer.step()
    model.load_state_dict(best_state)
    model.eval()
    return {'selected_step':selected,'executed_steps':steps,'validation_mse':best_loss,'trace':trace}

In [ ]:
if RUN_EXERCISES:
    try:
        for val_sign in (1.,-1.):
            net=torch.nn.Linear(1,1,bias=False);net.weight.data.zero_()
            xx=torch.ones(4,1);yy=torch.ones(4,1)
            result=fit_with_validation(net,xx,yy,xx,val_sign*yy,8,.1)
            assert len(result['trace'])==9 and result['executed_steps']==8
            assert result['selected_step']==(8 if val_sign==1 else 0)
            with torch.no_grad():actual=float(((net(xx)-val_sign*yy)**2).mean())
            assert np.isclose(actual,result['validation_mse'])
            assert np.isclose(actual,min(r['validation_mse'] for r in result['trace']))
        try:
            fit_with_validation(net,xx,yy,xx,yy,-1,.1)
            raise AssertionError('negative budget accepted')
        except ValueError:pass
        task_status[4]=True
    except NotImplementedError:
        task_status[4]=None
    except Exception as exc:
        task_status[4]=False
        print("Task 4 check failed:",type(exc).__name__,str(exc))
else:
    task_status[4]=None
print("Task 4:","PASS" if task_status[4] is True else "NOT SUBMITTED" if task_status[4] is None else "FAILED")

## Assessment: 100 points

Coding Tasks 1–4: [5, 5, 5, 25] points; derivation 25, interpretation 25, reproducibility 10. Automated checks assess only coding behavior.

Derive: derive permutation invariance of pooled sensor attention and distinguish cardinality dropout from mean imputation.

Worked interpretation: An attention set is equivariant before pooling and invariant afterward. Dropping tokens reduces cardinality, whereas replacing their values with zero retains a different input. Compare actual parameter counts, selection steps and validation criteria before attributing accuracy to architecture.

Reject and explain: “A model that reaches its update ceiling is necessarily converged, and equal update caps guarantee equal compute.”

An accurate numerical observation without its split, representation, or causal limitation earns at most half of interpretation points. Instructor marking and a timed student pilot are still required; automated checks establish interface/behavior only.



## Interpretation and submission

Submit the four completed functions, the requested plots/tables, and a 300-word claim ledger. Distinguish observations, fitted quantities, independent evaluation and unsupported extrapolation. Retain failed seeds. Use the lecture source for the rubric and assumptions.

In [ ]:
print("Coding tasks passed:",sum(v is True for v in task_status.values()),"/",len(task_status))
assert all(v is True for v in task_status.values())
workspace.cleanup()